# Why Don't Spectral Detectors Generalize Across Generators?

Two experiments to explain the cross-dataset failure (GenImage model → CIFAKE: MCC 0.05):

1. **Band-pass accuracy**: mask the spectrum into low/mid/high frequency bands and evaluate
   both models on each band separately. If the models rely on different bands, that explains
   why one can't do the other's job.

2. **Grad-CAM**: visualize which spectral regions each CNN2D model attends to on the same
   input images. Different attention = different learned features.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from tqdm import tqdm
from torch.utils.data import DataLoader
from sklearn.metrics import roc_auc_score, accuracy_score, matthews_corrcoef

from src.dataset import FrequencyDataset
from src.models import build_model

device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
DATA_DIR = '../data/processed/birdy654/cifake-real-and-ai-generated-synthetic-images/versions/3/test'
dataset = FrequencyDataset(root=DATA_DIR, size=224)
print(f'Dataset: {dataset.label_counts()}, total: {len(dataset)}')

In [ ]:
def load_2d_model(ckpt_path):
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    assert ckpt['model_type'] == '2d'
    model = build_model('2d').to(device)
    model.load_state_dict(ckpt['model_state'])
    model.eval()
    return model

model_cifake = load_2d_model('../results/cifake/best_2d.pt')
model_genimage = load_2d_model('../results/genimage/best_2d.pt')
print('Both CNN2D models loaded.')

## Part 1: Band-Pass Accuracy

We create radial masks to isolate frequency bands, then evaluate each model
on spectra where only one band is preserved (rest zeroed out).

| Band | Radial range | What it captures |
|------|-------------|------------------|
| Low  | r ≤ 20      | Overall brightness, large-scale structure |
| Mid  | 20 < r ≤ 60 | Textures, medium detail |
| High | r > 60      | Fine detail, edges, potential artifacts |

In [ ]:
H, W = 224, 224
cy, cx = H // 2, W // 2
y, x = np.ogrid[:H, :W]
r = np.sqrt((x - cx)**2 + (y - cy)**2)

R_LOW, R_MID = 20, 60

bands = {
    'full':       np.ones((H, W), dtype=np.float32),
    f'low (r≤{R_LOW})': (r <= R_LOW).astype(np.float32),
    f'mid ({R_LOW}<r≤{R_MID})': ((r > R_LOW) & (r <= R_MID)).astype(np.float32),
    f'high (r>{R_MID})': (r > R_MID).astype(np.float32),
}

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, (name, mask) in zip(axes, bands.items()):
    ax.imshow(mask, cmap='gray')
    ax.set_title(name)
    ax.axis('off')
plt.suptitle('Frequency band masks', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
band_masks_torch = {name: torch.from_numpy(m).to(device) for name, m in bands.items()}

def collate_fn(batch):
    spec2d, prof1d, labels = zip(*batch)
    return torch.stack(spec2d), torch.stack(prof1d), torch.tensor(labels)

loader = DataLoader(dataset, batch_size=64, shuffle=False, num_workers=4, collate_fn=collate_fn)

def evaluate_with_mask(model, loader, mask_tensor):
    all_labels, all_probs = [], []
    with torch.no_grad():
        for spec2d, _, labels in loader:
            masked = spec2d.to(device) * mask_tensor
            logits = model(masked)
            probs = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()
            all_probs.extend(probs)
            all_labels.extend(labels.numpy())
    all_labels = np.array(all_labels)
    all_probs = np.array(all_probs)
    preds = (all_probs >= 0.5).astype(int)
    return {
        'auc': roc_auc_score(all_labels, all_probs),
        'acc': accuracy_score(all_labels, preds),
        'mcc': matthews_corrcoef(all_labels, preds),
    }

results = {}
for model_name, model in [('CIFAKE-trained', model_cifake), ('GenImage-trained', model_genimage)]:
    results[model_name] = {}
    for band_name, mask_tensor in band_masks_torch.items():
        print(f'Evaluating {model_name} on {band_name}...')
        results[model_name][band_name] = evaluate_with_mask(model, loader, mask_tensor)

print('\nDone.')

In [ ]:
band_names = list(bands.keys())
model_names = list(results.keys())

print(f"{'Band':<20} | {'CIFAKE-trained':>40} | {'GenImage-trained':>40}")
print(f"{'':20} | {'AUC':>8} {'Acc':>8} {'MCC':>8}       | {'AUC':>8} {'Acc':>8} {'MCC':>8}")
print('-' * 105)
for band in band_names:
    rc = results['CIFAKE-trained'][band]
    rg = results['GenImage-trained'][band]
    print(f"{band:<20} | {rc['auc']:>8.4f} {rc['acc']:>8.4f} {rc['mcc']:>8.4f}       | {rg['auc']:>8.4f} {rg['acc']:>8.4f} {rg['mcc']:>8.4f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

x_pos = np.arange(len(band_names))
width = 0.35

for ax, metric in zip(axes, ['auc', 'acc', 'mcc']):
    vals_c = [results['CIFAKE-trained'][b][metric] for b in band_names]
    vals_g = [results['GenImage-trained'][b][metric] for b in band_names]

    bars_c = ax.bar(x_pos - width/2, vals_c, width, label='CIFAKE-trained', color='#2196F3')
    bars_g = ax.bar(x_pos + width/2, vals_g, width, label='GenImage-trained', color='#FF5722')

    ax.set_xticks(x_pos)
    ax.set_xticklabels(band_names, rotation=25, ha='right', fontsize=9)
    ax.set_title(metric.upper(), fontsize=13)
    ax.legend(fontsize=9)

    if metric == 'mcc':
        ax.set_ylim(-0.2, 1.0)
        ax.axhline(0, color='gray', ls='--', lw=0.8)
    else:
        ax.set_ylim(0.4, 1.0)
        ax.axhline(0.5, color='gray', ls='--', lw=0.8)

plt.suptitle('Band-Pass Accuracy: Which frequency bands does each model rely on?\n(evaluated on CIFAKE test set)', fontsize=14)
plt.tight_layout()
plt.savefig('../figures/band_pass_accuracy.png', dpi=150, bbox_inches='tight')
plt.show()

## Part 2: Grad-CAM on CNN2D

Grad-CAM highlights which regions of the 2D spectrum the model uses for its prediction.
Since the input is a centered log-power spectrum (DC at center, high freq at edges),
the heatmap directly shows which spatial frequencies matter.

We compare both models on the same fake image — if they attend to different spectral regions,
that visually explains the generalization failure.

In [ ]:
class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.activations = None
        self.gradients = None
        target_layer.register_forward_hook(self._save_activation)
        target_layer.register_full_backward_hook(self._save_gradient)

    def _save_activation(self, module, input, output):
        self.activations = output.detach()

    def _save_gradient(self, module, grad_input, grad_output):
        self.gradients = grad_output[0].detach()

    def __call__(self, x, target_class=None):
        self.model.zero_grad()
        output = self.model(x)
        if target_class is None:
            target_class = output.argmax(dim=1)
        one_hot = torch.zeros_like(output)
        one_hot.scatter_(1, target_class.unsqueeze(1), 1.0)
        output.backward(gradient=one_hot, retain_graph=True)

        weights = self.gradients.mean(dim=(2, 3), keepdim=True)
        cam = (weights * self.activations).sum(dim=1, keepdim=True)
        cam = F.relu(cam)
        cam = F.interpolate(cam, size=x.shape[2:], mode='bilinear', align_corners=False)
        cam = cam.squeeze()
        if cam.max() > 0:
            cam = cam / cam.max()
        return cam.cpu().numpy(), output

gradcam_cifake = GradCAM(model_cifake, model_cifake.features[-1])
gradcam_genimage = GradCAM(model_genimage, model_genimage.features[-1])
print('Grad-CAM hooks registered on last conv block of each model.')

In [ ]:
np.random.seed(42)
fake_indices = [i for i, (_, label) in enumerate(dataset.samples) if label == 1]
real_indices = [i for i, (_, label) in enumerate(dataset.samples) if label == 0]
sample_indices = np.random.choice(fake_indices, 4, replace=False).tolist() + \
                 np.random.choice(real_indices, 2, replace=False).tolist()

n = len(sample_indices)
fig, axes = plt.subplots(n, 5, figsize=(25, 4 * n))

for row, idx in enumerate(sample_indices):
    spec2d, prof1d, label = dataset[idx]
    x = spec2d.unsqueeze(0).to(device).requires_grad_(True)

    cam_c, out_c = gradcam_cifake(x, target_class=torch.tensor([1]).to(device))
    x2 = spec2d.unsqueeze(0).to(device).requires_grad_(True)
    cam_g, out_g = gradcam_genimage(x2, target_class=torch.tensor([1]).to(device))

    prob_c = torch.softmax(out_c, dim=1)[0, 1].item()
    prob_g = torch.softmax(out_g, dim=1)[0, 1].item()

    spec_np = spec2d.squeeze().numpy()
    truth = 'FAKE' if label == 1 else 'REAL'

    axes[row, 0].imshow(spec_np, cmap='inferno')
    axes[row, 0].set_title(f'{truth} — Log-Power Spectrum')

    axes[row, 1].imshow(cam_c, cmap='jet', vmin=0, vmax=1)
    axes[row, 1].set_title(f'CIFAKE model Grad-CAM\nP(fake)={prob_c:.3f}')

    axes[row, 2].imshow(spec_np, cmap='inferno')
    axes[row, 2].imshow(cam_c, cmap='jet', alpha=0.5, vmin=0, vmax=1)
    axes[row, 2].set_title('CIFAKE Grad-CAM overlay')

    axes[row, 3].imshow(cam_g, cmap='jet', vmin=0, vmax=1)
    axes[row, 3].set_title(f'GenImage model Grad-CAM\nP(fake)={prob_g:.3f}')

    axes[row, 4].imshow(spec_np, cmap='inferno')
    axes[row, 4].imshow(cam_g, cmap='jet', alpha=0.5, vmin=0, vmax=1)
    axes[row, 4].set_title('GenImage Grad-CAM overlay')

for ax in axes.ravel():
    ax.axis('off')

plt.suptitle('Grad-CAM: Where in the spectrum does each model look?\n(CIFAKE test images — same input, different models)', fontsize=15, y=1.01)
plt.tight_layout()
plt.savefig('../figures/gradcam_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## Part 3: Mean Grad-CAM across classes

Average Grad-CAM heatmaps over many images to see the overall spectral attention pattern
for each model — removes per-image noise and reveals systematic differences.

In [ ]:
N_SAMPLES = 200
np.random.seed(42)
sample_fake = np.random.choice(fake_indices, min(N_SAMPLES, len(fake_indices)), replace=False)
sample_real = np.random.choice(real_indices, min(N_SAMPLES, len(real_indices)), replace=False)

def mean_gradcam(gradcam_fn, indices, dataset):
    accum = np.zeros((224, 224), dtype=np.float64)
    for idx in tqdm(indices, desc='Grad-CAM', leave=False):
        spec2d, _, _ = dataset[idx]
        x = spec2d.unsqueeze(0).to(device).requires_grad_(True)
        cam, _ = gradcam_fn(x, target_class=torch.tensor([1]).to(device))
        accum += cam
    return (accum / len(indices)).astype(np.float32)

print('Computing mean Grad-CAM maps (this takes a few minutes)...')
mean_cam_cifake_fake = mean_gradcam(gradcam_cifake, sample_fake, dataset)
mean_cam_genimage_fake = mean_gradcam(gradcam_genimage, sample_fake, dataset)
mean_cam_cifake_real = mean_gradcam(gradcam_cifake, sample_real, dataset)
mean_cam_genimage_real = mean_gradcam(gradcam_genimage, sample_real, dataset)
print('Done.')

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 11))

vmax = max(mean_cam_cifake_fake.max(), mean_cam_genimage_fake.max(),
           mean_cam_cifake_real.max(), mean_cam_genimage_real.max())

for col, (cam_fake, cam_real, title) in enumerate([
    (mean_cam_cifake_fake, mean_cam_cifake_real, 'CIFAKE-trained model'),
    (mean_cam_genimage_fake, mean_cam_genimage_real, 'GenImage-trained model'),
]):
    im0 = axes[0, col].imshow(cam_fake, cmap='jet', vmin=0, vmax=vmax)
    axes[0, col].set_title(f'{title}\nMean Grad-CAM on FAKE images')
    axes[0, col].axis('off')

    im1 = axes[1, col].imshow(cam_real, cmap='jet', vmin=0, vmax=vmax)
    axes[1, col].set_title(f'{title}\nMean Grad-CAM on REAL images')
    axes[1, col].axis('off')

diff = mean_cam_cifake_fake - mean_cam_genimage_fake
dmax = np.abs(diff).max()
im_d = axes[0, 2].imshow(diff, cmap='RdBu_r', vmin=-dmax, vmax=dmax)
axes[0, 2].set_title('Attention difference (CIFAKE − GenImage)\non FAKE images\nRed = CIFAKE attends more')
axes[0, 2].axis('off')
plt.colorbar(im_d, ax=axes[0, 2], fraction=0.046)

diff_r = mean_cam_cifake_real - mean_cam_genimage_real
dmax_r = np.abs(diff_r).max()
im_dr = axes[1, 2].imshow(diff_r, cmap='RdBu_r', vmin=-dmax_r, vmax=dmax_r)
axes[1, 2].set_title('Attention difference (CIFAKE − GenImage)\non REAL images\nRed = CIFAKE attends more')
axes[1, 2].axis('off')
plt.colorbar(im_dr, ax=axes[1, 2], fraction=0.046)

plt.suptitle('Mean Grad-CAM: Systematic attention patterns of each model', fontsize=15, y=1.02)
plt.tight_layout()
plt.savefig('../figures/mean_gradcam_comparison.png', dpi=150, bbox_inches='tight')
plt.show()